# Phase 3.4 — Hybrid Recommendation Engine

Combine collaborative, content, popularity, and recency signals into one practical Top-K recommender.

This notebook intentionally uses a small candidate pool and a deterministic evaluation sample so the hybrid layer remains fast enough for development.


In [1]:
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_UI_PATH = PROCESSED_DIR / "train_user_item.csv"
TRAIN_PATH = PROCESSED_DIR / "train_interactions.csv"
TEST_PATH = PROCESSED_DIR / "test_interactions.csv"

PROPERTIES_1 = RAW_DIR / "item_properties_part1.csv"
PROPERTIES_2 = RAW_DIR / "item_properties_part2.csv"

K = 10
CANDIDATES_PER_SIGNAL = 30

print("Project root:", PROJECT_ROOT)


Project root: f:\annuspeaks.com\recommendation-system


## 3.4.1 Load Training Data and Build Popularity / Recency Signals


In [2]:
train_ui = pd.read_csv(
    TRAIN_UI_PATH,
    usecols=["user_id", "item_id", "total_weight", "last_timestamp"],
)

train = pd.read_csv(
    TRAIN_PATH,
    usecols=["user_id", "item_id", "weight", "timestamp"],
)

test = pd.read_csv(
    TEST_PATH,
    usecols=["user_id", "item_id"],
)

# Popularity score from training data only.
popularity = (
    train_ui.groupby("item_id")["total_weight"]
    .sum()
    .sort_values(ascending=False)
)

popular_items = popularity.index.to_numpy()

# Global recency: latest observed interaction for each product.
item_recency = (
    train_ui.groupby("item_id")["last_timestamp"]
    .max()
)

print("Training user-item pairs:", f"{len(train_ui):,}")
print("Catalog items:", f"{len(popular_items):,}")


Training user-item pairs: 1,939,777
Catalog items: 228,392


## 3.4.2 Collaborative Candidate Generator

Use the Phase 3.3 matrix-factorization approach, but score only the candidate pool rather than ranking the entire catalog for every final recommendation.


In [3]:
# Build sparse user-item matrix.

user_ids = train_ui["user_id"].unique()
item_ids = train_ui["item_id"].unique()

user_to_idx = {x: i for i, x in enumerate(user_ids)}
item_to_idx = {x: i for i, x in enumerate(item_ids)}
idx_to_item = np.asarray(item_ids)

rows = train_ui["user_id"].map(user_to_idx).to_numpy()
cols = train_ui["item_id"].map(item_to_idx).to_numpy()
values = train_ui["total_weight"].to_numpy(dtype=np.float32)

ui_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(item_ids)),
    dtype=np.float32,
)

n_components = min(32, min(ui_matrix.shape) - 1)

svd = TruncatedSVD(
    n_components=n_components,
    algorithm="randomized",
    n_iter=3,
    random_state=42,
)

user_factors = svd.fit_transform(ui_matrix)
item_factors = svd.components_.T

print("Matrix:", ui_matrix.shape)
print("Latent dimensions:", n_components)


Matrix: (1407580, 228392)
Latent dimensions: 32


In [4]:
# User interaction history for repetition filtering.

seen_items = defaultdict(set)

for row in train_ui.itertuples(index=False):
    seen_items[row.user_id].add(row.item_id)

def collaborative_candidates(user_id, n=CANDIDATES_PER_SIGNAL):
    if user_id not in user_to_idx:
        return []

    scores = item_factors @ user_factors[user_to_idx[user_id]]

    # Only obtain a manageable candidate pool.
    pool_size = min(n + len(seen_items[user_id]) + 20, len(scores))
    top_idx = np.argpartition(scores, -pool_size)[-pool_size:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

    result = []
    for idx in top_idx:
        item_id = idx_to_item[idx]
        if item_id in seen_items[user_id]:
            continue
        result.append((item_id, float(scores[idx])))
        if len(result) >= n:
            break

    return result


## 3.4.3 Content Candidate Generator

Build a compact TF-IDF representation from the available generic product metadata and use recent user items as content seeds.


In [5]:
# Build product metadata text.

parts = []

for path in [PROPERTIES_1, PROPERTIES_2]:
    for chunk in pd.read_csv(
        path,
        usecols=["itemid", "property", "value"],
        chunksize=250_000,
    ):
        chunk = chunk.dropna(subset=["itemid", "property", "value"]).copy()
        chunk["property"] = chunk["property"].astype(str).str.strip()
        chunk["value"] = chunk["value"].astype(str).str.strip()
        chunk["token"] = chunk["property"] + "=" + chunk["value"]
        parts.append(chunk[["itemid", "token"]])

properties = pd.concat(parts, ignore_index=True).drop_duplicates()

product_text = (
    properties.groupby("itemid")["token"]
    .agg(" ".join)
    .reset_index()
    .rename(columns={"itemid": "item_id"})
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+=\S+\b",
    min_df=2,
    max_features=50_000,
)

product_matrix = vectorizer.fit_transform(product_text["token"])

content_item_to_idx = {
    item_id: i
    for i, item_id in enumerate(product_text["item_id"])
}

content_items = product_text["item_id"].to_numpy()

content_index = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=31,
    n_jobs=-1,
)

content_index.fit(product_matrix)

print("Content products:", f"{len(content_items):,}")
print("TF-IDF shape:", product_matrix.shape)


Content products: 417,053
TF-IDF shape: (417053, 50000)


In [6]:
# Recent histories only for users used in hybrid evaluation.
# Avoid sorting/grouping the entire 2.3M-row interaction table.

test_targets = (
    test.groupby("user_id")["item_id"]
    .last()
    .to_dict()
)

MAX_EVAL_USERS = 100
eval_users = sorted(test_targets)[:MAX_EVAL_USERS]
eval_user_set = set(eval_users)

eval_train = train[train["user_id"].isin(eval_user_set)].copy()

recent_history = (
    eval_train
    .sort_values(["user_id", "timestamp"], kind="mergesort")
    .groupby("user_id")["item_id"]
    .apply(lambda s: s.drop_duplicates().tail(10).tolist())
    .to_dict()
)

# Cache content candidates because the same users are used by
# recommendation, evaluation, and diversity checks.
content_candidate_cache = {}

def content_candidates(user_id, n=CANDIDATES_PER_SIGNAL):
    cache_key = (user_id, n)

    if cache_key in content_candidate_cache:
        return content_candidate_cache[cache_key]

    history = recent_history.get(user_id, [])
    candidates = {}

    for item_id in reversed(history[-5:]):
        if item_id not in content_item_to_idx:
            continue

        idx = content_item_to_idx[item_id]
        distances, indices = content_index.kneighbors(
            product_matrix[idx],
            n_neighbors=31,
        )

        for distance, neighbor_idx in zip(distances[0], indices[0]):
            candidate = content_items[neighbor_idx]

            if candidate == item_id or candidate in seen_items[user_id]:
                continue

            similarity = 1.0 - float(distance)
            candidates[candidate] = max(
                candidates.get(candidate, 0.0),
                similarity,
            )

    result = sorted(
        candidates.items(),
        key=lambda x: x[1],
        reverse=True
    )[:n]

    content_candidate_cache[cache_key] = result
    return result

print("Evaluation users:", len(eval_users))
print("Evaluation training rows:", f"{len(eval_train):,}")
print("Recent histories built:", len(recent_history))


Evaluation users: 100
Evaluation training rows: 461
Recent histories built: 100


## 3.4.4 Hybrid Candidate Generation and Ranking

Candidate sources:
- Collaborative filtering
- Content similarity
- Popularity
- Recency

The final score is a weighted combination of normalized signal scores.


In [7]:
def minmax_scores(values):
    if not values:
        return {}

    scores = np.asarray(list(values.values()), dtype=float)
    lo, hi = scores.min(), scores.max()

    if hi == lo:
        return {key: 1.0 for key in values}

    return {
        key: (value - lo) / (hi - lo)
        for key, value in values.items()
    }

def hybrid_recommend(user_id, k=10):
    candidates = set()

    cf = dict(collaborative_candidates(user_id))
    cb = dict(content_candidates(user_id))

    candidates.update(cf.keys())
    candidates.update(cb.keys())

    # Add a small popularity fallback pool.
    candidates.update(
        item_id
        for item_id in popular_items[:CANDIDATES_PER_SIGNAL]
        if item_id not in seen_items[user_id]
    )

    if not candidates:
        return popular_items[:k].tolist()

    # Normalize each signal only over this user's candidate pool.
    cf_norm = minmax_scores({
        item_id: cf.get(item_id, 0.0)
        for item_id in candidates
    })

    cb_norm = minmax_scores({
        item_id: cb.get(item_id, 0.0)
        for item_id in candidates
    })

    popularity_raw = {
        item_id: float(popularity.get(item_id, 0.0))
        for item_id in candidates
    }
    popularity_norm = minmax_scores(popularity_raw)

    recency_raw = {
        item_id: float(item_recency.get(item_id, 0))
        for item_id in candidates
    }
    recency_norm = minmax_scores(recency_raw)

    # Practical starting weights.
    final_scores = {}

    for item_id in candidates:
        final_scores[item_id] = (
            0.45 * cf_norm[item_id]
            + 0.30 * cb_norm[item_id]
            + 0.15 * popularity_norm[item_id]
            + 0.10 * recency_norm[item_id]
        )

    ranked = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    # Basic repetition control: remove duplicate IDs and return Top-K.
    recommendations = []
    for item_id, score in ranked:
        if item_id in seen_items[user_id]:
            continue

        recommendations.append({
            "item_id": item_id,
            "score": float(score),
        })

        if len(recommendations) == k:
            break

    return recommendations


In [8]:
# Example hybrid recommendation.

example_user = int(train_ui["user_id"].iloc[0])

print("User:", example_user)
display(pd.DataFrame(hybrid_recommend(example_user, K)))


User: 0


,item_id,score
0,20388,0.563134
1,369447,0.406569
2,45786,0.399673
3,404151,0.383492
4,169503,0.377379
5,8692,0.350637
6,7943,0.318501
7,320130,0.300076
8,35327,0.295767
9,9877,0.295649


## 3.4.5 Evaluate Hybrid Recommendation

Use a small deterministic test sample. The evaluation uses the same held-out test target setup as earlier phases.


In [9]:
# Evaluate the hybrid recommender on the fixed evaluation users.

hits = 0
evaluated = 0

for user_id in eval_users:
    recommendations = hybrid_recommend(user_id, K)
    recommendation_ids = {row["item_id"] for row in recommendations}

    if test_targets[user_id] in recommendation_ids:
        hits += 1

    evaluated += 1

hybrid_hit_rate = hits / evaluated if evaluated else 0.0

hybrid_result = pd.DataFrame([{
    "model": "Hybrid",
    "K": K,
    "evaluated_users": evaluated,
    "hits": hits,
    "HitRate@10": hybrid_hit_rate,
}])

display(hybrid_result)


,model,K,evaluated_users,hits,HitRate@10
0,Hybrid,10,100,2,0.02


## 3.4.6 Basic Diversity and Repetition Controls

Measure whether the final recommendations contain repeated products and how many unique products are exposed across the evaluation sample.


In [10]:
recommendation_lists = []

for user_id in eval_users:
    recs = hybrid_recommend(user_id, K)
    recommendation_lists.append(
        [row["item_id"] for row in recs]
    )

all_recommendations = [
    item_id
    for recs in recommendation_lists
    for item_id in recs
]

total_recommendations = len(all_recommendations)
unique_recommendations = len(set(all_recommendations))

repetition_rate = (
    1 - unique_recommendations / total_recommendations
    if total_recommendations
    else 0.0
)

print("Total recommendations:", total_recommendations)
print("Unique recommended products:", unique_recommendations)
print("Repetition rate:", f"{repetition_rate:.4f}")


Total recommendations: 1000
Unique recommended products: 558
Repetition rate: 0.4420


## 3.4.7 Select the Practical Hybrid Approach

The current hybrid uses a simple weighted score:

`0.45 collaborative + 0.30 content + 0.15 popularity + 0.10 recency`

These are development weights, not learned ranking weights. They provide a transparent starting point for the final ranking layer.


In [11]:
# Final validation.

assert evaluated > 0
assert 0 <= hybrid_hit_rate <= 1
assert total_recommendations == len(all_recommendations)
assert unique_recommendations <= total_recommendations

print("Phase 3.4 validation: PASS")
print("Hybrid HitRate@10:", f"{hybrid_hit_rate:.4f}")
print("Recommendation repetition rate:", f"{repetition_rate:.4f}")
print("Weights: CF=0.45, Content=0.30, Popularity=0.15, Recency=0.10")


Phase 3.4 validation: PASS
Hybrid HitRate@10: 0.0200
Recommendation repetition rate: 0.4420
Weights: CF=0.45, Content=0.30, Popularity=0.15, Recency=0.10
